# XMM-SAS 19 validation: merge EPIC spectra

This is a functionality introduced as prototype in SAS 18, tasks [`epicspeccombine`](https://xmm-tools.cosmos.esa.int/external/sas/current/doc/epicspeccombine/index.html). The usage is explained in [this thread](https://www.cosmos.esa.int/web/xmm-newton/sas-thread-epic-merging).

This notebook will process  the Circinus Galaxy

_Ivan Valtchanov_, Sep 2020

## Set up

* Working on my Linux desktop `xmml72`.
* Source `HEADAS` and `XMM_SAS`, using a shell script `start_sas19.sh`.
* Scripts/notebooks and this report are in folder `~/Dropbox/Works/XMM/sasval19`.

## Testing dataset

I want to compare the results with those presented in the data analysis thread. Neither H1426+428, `OBS_ID=0310190101` nor the Circinus Galaxy, `OBS_ID=0111240101` are included in the validation sets. So, I have two options: use the PPS with SAS18 or reprocess with SAS19. The easiest is to use the currently available SAS18 pipeline products. 

**Note:** [The thread](https://www.cosmos.esa.int/web/xmm-newton/sas-thread-epic-merging) does not provide any details on the source and background regions. So, I use my own selection.

## Testing procedure

1. Download the PPS products for `OBS_ID` using [`astroquery.esa.xmm_newton`](https://astroquery.readthedocs.io/en/latest/esa/xmm_newton.html).
2. Use the image in band 8 to select the source and background regions and save it in **Equatorial coordinates**. This is outside the notebook, interactive work.
3. Filter the event lists for GTI and FLAGs.
4. Use the cleaned event lists from step 3 to extract source and background spectra with expression using `(RA,DEC)` and run `backscale`.
5. Generate RMF and ARF. Note that special spectral channel selection and spectral bin size is necessary. As well as a special switch in `rmfgen` for pn.
5. Run `epicspeccombine` with the generated spectra.

Alternative generation of spectra with [`multiespecget`](https://xmm-tools.cosmos.esa.int/external/sas/current/doc/multiespecget/index.html) was not tested.

## Outcome:

The tests were successful. The tested task `epicspeccombine` behaved as expected.

### Cooments:
* If the RMFs were not generated as expained in the thread (i.e. initially I did not use `withenergybins=yes energymin=0.1 energymax=12.0 nenergybins=2400`) then the task fails with Segmentation fault (core dumped) and the error message is not helpful to understand why it failed. In addition, it kept some intermediate files with names `TEST_rsp_0.ds`, `TEST_rsp_1.ds`, `TEST_rsp_2.ds`.

_Ivan Valtchanov_, 29 Sep 2020


In [1]:
import os
import subprocess
import sys
#import requests
import tarfile
import logging
import glob

import numpy as np

from astropy.io import fits
from astroquery.esa.xmm_newton import XMMNewton as xmm

%matplotlib inline

import matplotlib.pylab as plt

#from astropy.wcs import WCS
#from astropy.coordinates import SkyCoord
#from astropy import units as u
#from astropy.visualization import simple_norm, PercentileInterval, ImageNormalize, ManualInterval
#from astropy.nddata import Cutout2D

#from regions import CircleSkyRegion

#import pysas



Created TAP+ (v1.2.1) - Connection:
	Host: nxsa.esac.esa.int
	Use HTTPS: False
	Port: 80
	SSL Port: 443


## Import some tools

One such tool is the execution of shell commands" `run_command()`

In [2]:
sasval19_dir = os.path.expanduser('~') + "/Dropbox/Work/XMM/xmmpy/sasval19"
sys.path.append(sasval19_dir)

In [5]:
import sasval19_tools as st19
#import importlib
#importlib.reload(st19)

## Set up XMM-SAS 19

Apparently having SAS in the terminal from where I started jupyterlab is not sufficient to have it available in the notebook, so I have to set this up

In [6]:
# sas_dir = '/sasbuild/installed/sasbld03n/GNU_CC_CXX_9.2.0/xmmsas_20200817_0927'
# os.environ["SAS_DIR"]= sas_dir
# os.environ["SAS_PATH"]=os.environ["SAS_DIR"]
# #os.environ["SAS_CCFPATH"]= "/ccf/valid"
# os.environ["SAS_CCFPATH"]= "/xdata/ccf/pub"
# #
# os.environ["SAS_VERBOSITY"]="4"
# os.environ["SAS_SUPPRESS_WARNING"]="1"
# path = os.environ["PATH"]
# os.environ["PATH"] = f"{sas_dir}/bin:{sas_dir}/binextra:{path}"

## Set up the input/output folders

The input folder will be the place where the PPS products are extracted. 

In [11]:
#
# set up the folders
#
rootDir = '/lhome/ivaltchanov/XMM/sasval19/spec_merge'
target = "circinus"
obsid="0111240101"
#target = "H1426+428"
#target = "h1426"
#obsid="0310190101"
#
wdir = f"{rootDir}/{target}"
out_dir = f"{wdir}/{obsid}/works"
pps_dir = f"{wdir}/{obsid}/pps"

In [12]:
#
# download the PPS
#
pps_tar_file = f"{wdir}/{obsid}_PPS_nxsa"
xmm.download_data(obsid,level="PPS",extension="FTZ",filename=pps_tar_file)

INFO: Copying file to /lhome/ivaltchanov/XMM/sasval19/spec_merge/circinus/0111240101_PPS_nxsa.tar... [astroquery.esa.xmm_newton.core]


In [14]:
# extract the PPS
#
pps_tar_file = f"{wdir}/{obsid}_PPS_nxsa.tar"
with tarfile.open(pps_tar_file,'r') as tar:
    tar.extractall(path=wdir)
print (f"PPS tar file extracted to folder {wdir}")

PPS tar file extracted to folder /lhome/ivaltchanov/XMM/sasval19/spec_merge/circinus


In [15]:
#
if (not os.path.isdir(pps_dir)):
    print (f"After tar extraction, no PPS folder found: {pps_dir}")
    raise FileNotFoundError
#
# Go to the PPS folder where we'll store all results
#
os.chdir(pps_dir)

## Set up the logging file

This file will hold all stdin/stdout/stderr outputs.

In [16]:
logging.basicConfig(level=logging.DEBUG,
                    format='%(asctime)s %(levelname)s %(message)s',
                    filename=f'{out_dir}/spec_merge_tests_06-10-2020.log',
                    filemode='w')


In [17]:
#
# and find and set the SAS_CCF

idx = glob.glob('*OBX000CALIND*.FTZ')
if len(idx) != 1:
    print (f'CCF file with pattern OBX000CALIND nod found in {pps_dir}')
    raise FileNotFoundError
#
os.environ["SAS_CCF"]= idx[0]
print (f"SAS_CCF set to {os.environ['SAS_CCF']}")
logging.info (f"SAS_CCF set to {os.environ['SAS_CCF']}")
#

SAS_CCF set to P0111240101OBX000CALIND0000.FTZ


In [18]:
#
# now get the available event lists an assign the filenames
#
evl = {}
evlists = glob.glob('*IEVLI*.FTZ')
for q in evlists:
    if ('M1' in q):
        evl['m1'] = q
    elif ('M2' in q):
        evl['m2'] = q
    elif ('PN' in q):
        evl['pn'] = q
    else:
        print (f'Cannot assign event list to EPIC: {q}')
        logger.warning(f'Cannot assign event list to EPIC: {q}')

## Use the PPS light curve to build a GTI

There is already a light-curve in the PPS products (`flare_file` in my code below) and we shall reuse it to derive the good-time-intervals. The threshold, derived by the pipeline, is stored in the FITS header metadata `FLCUTTHR`, and I will read it and use it.

In [19]:
flare_files = glob.glob("*FBKTSR0000.FTZ")

gti_set = {}
for xf in flare_files:
    #
    # extract the threshold for flares as derived by the pipeline
    #
    with fits.open(xf) as hdu:
        if ('M1' in xf):
            inst = 'm1'
        elif ('M2' in xf):
            inst = 'm2'
        elif ('PN' in xf):
            inst = 'pn'
        else:
            continue
        fcut = hdu['RATE'].header['FLCUTTHR']
        #
        #
        # generate a GTI file
        #
        gti_set[inst] = f"{out_dir}/{inst}_gti.fits"
        gti_command = f'tabgtigen table=\'{xf}\' expression=\'RATE<={fcut}\' gtiset=\'{gti_set[inst]}\''
        #
        #print (gti_command)
        status,_ = st19.run_command(gti_command)
        if (status != 0):
            raise RuntimeError


Execution of tabgtigen table='P0111240101PNS003FBKTSR0000.FTZ' expression='RATE<=6.84935474' gtiset='/lhome/ivaltchanov/XMM/sasval19/spec_merge/circinus/0111240101/works/pn_gti.fits' returned 0
Execution of tabgtigen table='P0111240101M2S002FBKTSR0000.FTZ' expression='RATE<=1.94254386' gtiset='/lhome/ivaltchanov/XMM/sasval19/spec_merge/circinus/0111240101/works/m2_gti.fits' returned 0
Execution of tabgtigen table='P0111240101M1S001FBKTSR0000.FTZ' expression='RATE<=2.25346661' gtiset='/lhome/ivaltchanov/XMM/sasval19/spec_merge/circinus/0111240101/works/m1_gti.fits' returned 0


## Clean the event lists for background using the PPS derived GTI

In [20]:
evl_clean = {}
for inst in ['m1','m2','pn']:
    evl_clean[inst] = f"{out_dir}/{inst}_clean_evlist.fits"
    if (inst == 'pn'):
        comm = f'evselect table={evl[inst]} expression="(FLAG==0) && (PATTERN<=4) && gti({gti_set[inst]},TIME)"' + \
            f' withfilteredset=yes filteredset={evl_clean[inst]} destruct=yes keepfilteroutput=yes'
    else:
        comm = f'evselect table={evl[inst]} expression="#XMMEA_EM && (PATTERN<=12) && gti({gti_set[inst]},TIME)"' + \
            f' withfilteredset=yes filteredset={evl_clean[inst]} destruct=yes keepfilteroutput=yes'
    status,_ = st19.run_command(comm)
    if (status != 0):
        raise RuntimeError

Execution of evselect table=P0111240101M1S001MIEVLI0000.FTZ expression="#XMMEA_EM && (PATTERN<=12) && gti(/lhome/ivaltchanov/XMM/sasval19/spec_merge/circinus/0111240101/works/m1_gti.fits,TIME)" withfilteredset=yes filteredset=/lhome/ivaltchanov/XMM/sasval19/spec_merge/circinus/0111240101/works/m1_clean_evlist.fits destruct=yes keepfilteroutput=yes returned 0
Execution of evselect table=P0111240101M2S002MIEVLI0000.FTZ expression="#XMMEA_EM && (PATTERN<=12) && gti(/lhome/ivaltchanov/XMM/sasval19/spec_merge/circinus/0111240101/works/m2_gti.fits,TIME)" withfilteredset=yes filteredset=/lhome/ivaltchanov/XMM/sasval19/spec_merge/circinus/0111240101/works/m2_clean_evlist.fits destruct=yes keepfilteroutput=yes returned 0
Execution of evselect table=P0111240101PNS003PIEVLI0000.FTZ expression="(FLAG==0) && (PATTERN<=4) && gti(/lhome/ivaltchanov/XMM/sasval19/spec_merge/circinus/0111240101/works/pn_gti.fits,TIME)" withfilteredset=yes filteredset=/lhome/ivaltchanov/XMM/sasval19/spec_merge/circinus/0

## Set up source and background regions

I've defined those interactively using `ds9`, the observation is in Small Window mode, so the same background cannot be used for MOS and PN.


In [21]:
#
# all 
#circle(213.29317,-65.34043,30")
#circle(213.15254,-65.484044,54") # background
#
radius = 30.0/3600.0 # arcsec source radius
radius2 = 54.0/3600.0 # arcsec background radius
src = {}
bkg = {}
#
src['m1'] = f'((RA,DEC) IN CIRCLE(213.29317,-65.34043,{radius}))'
src['m2'] = f'((RA,DEC) IN CIRCLE(213.29317,-65.34043,{radius}))'
src['pn'] = f'((RA,DEC) IN CIRCLE(213.29317,-65.34043,{radius}))'
#
bkg['m1'] = f'((RA,DEC) IN CIRCLE(213.15254,-65.484044,{radius2}))'
bkg['m2'] = f'((RA,DEC) IN CIRCLE(213.15254,-65.484044,{radius2}))'
bkg['pn'] = f'((RA,DEC) IN CIRCLE(213.15254,-65.484044,{radius2}))'
#
#
special_expr = "withspectrumset=yes spectralbinsize=5 specchannelmin=0 specchannelmax=11999 energycolumn=PI"
#
for inst in ['m1','m2','pn']:
    spec_src_set = f"{out_dir}/{inst}_src_spec.fits"
    comm = f"evselect table={evl_clean[inst]} expression=\"{src[inst]}\" spectrumset={spec_src_set} " + special_expr
    status,_ = st19.run_command(comm)
    if (status != 0):
        raise RuntimeError
    #
    # now run backscale
    comm = f'backscale spectrumset={spec_src_set} badpixlocation={evl[inst]}'
    status,_ = st19.run_command(comm)
    if (status != 0):
        raise RuntimeError
    #
    # the background region
    spec_bkg_set = f"{out_dir}/{inst}_bkg_spec.fits"
    comm = f"evselect table={evl_clean[inst]} expression=\"{bkg[inst]}\" spectrumset={spec_bkg_set} " + special_expr
    status,_ = st19.run_command(comm)
    if (status != 0):
        raise RuntimeError
    #
    # now run backscale
    comm = f'backscale spectrumset={spec_bkg_set} badpixlocation={evl[inst]}'
    status,_ = st19.run_command(comm)
    if (status != 0):
        raise RuntimeError


Execution of evselect table=/lhome/ivaltchanov/XMM/sasval19/spec_merge/circinus/0111240101/works/m1_clean_evlist.fits expression="((RA,DEC) IN CIRCLE(213.29317,-65.34043,0.008333333333333333))" spectrumset=/lhome/ivaltchanov/XMM/sasval19/spec_merge/circinus/0111240101/works/m1_src_spec.fits withspectrumset=yes spectralbinsize=5 specchannelmin=0 specchannelmax=11999 energycolumn=PI returned 0
Execution of backscale spectrumset=/lhome/ivaltchanov/XMM/sasval19/spec_merge/circinus/0111240101/works/m1_src_spec.fits badpixlocation=P0111240101M1S001MIEVLI0000.FTZ returned 0
Execution of evselect table=/lhome/ivaltchanov/XMM/sasval19/spec_merge/circinus/0111240101/works/m1_clean_evlist.fits expression="((RA,DEC) IN CIRCLE(213.15254,-65.484044,0.015))" spectrumset=/lhome/ivaltchanov/XMM/sasval19/spec_merge/circinus/0111240101/works/m1_bkg_spec.fits withspectrumset=yes spectralbinsize=5 specchannelmin=0 specchannelmax=11999 energycolumn=PI returned 0
Execution of backscale spectrumset=/lhome/iva

## Generate RMF and ARF

In [26]:
#
os.chdir(pps_dir)
keepPrevious = False
for inst in ['m1','m2','pn']:
    spec_src_set = f"{out_dir}/{inst}_src_spec.fits"
    rmfset = f"{out_dir}/{inst}_src_spec.rmf"
    arfset = f"{out_dir}/{inst}_src_spec.arf"
    #
    if (os.path.isfile(rmfset) and os.path.isfile(arfset) and keepPrevious):
        print (f"Skipping RMF and ARF generation for {inst} as files already exist")
        continue
    #
    # wrong way
    #comm = f"rmfgen spectrumset={spec_src_set} rmfset={rmfset}"
    # good way
    comm = f"rmfgen spectrumset={spec_src_set} rmfset={rmfset} withenergybins=yes energymin=0.1 energymax=12.0 nenergybins=2400"
    if (inst == 'pn'):
        # wrong way
        #comm = f"rmfgen spectrumset={spec_src_set} rmfset={rmfset} acceptchanrange=yes"
        # good way
        comm = f"rmfgen spectrumset={spec_src_set} rmfset={rmfset} acceptchanrange=yes withenergybins=yes energymin=0.1 energymax=12.0 nenergybins=2400"
    print ('Running rmfgen, can take time...')
    status,_ = st19.run_command(comm)
    if (status != 0):
        raise RuntimeError
    # now generate the ARF
    comm = f"arfgen spectrumset={spec_src_set} withrmfset=yes rmfset={rmfset} badpixlocation={evl[inst]} arfset={arfset}"
    if (inst == 'pn'):
        comm = f"arfgen spectrumset={spec_src_set} withrmfset=yes rmfset={rmfset} badpixlocation={evl[inst]} arfset={arfset}"
    print ('Running arfgen, can take time...')
    status,_ = st19.run_command(comm)
    if (status != 0):
        raise RuntimeError


Running rmfgen, can take time...


Execution of rmfgen spectrumset=/lhome/ivaltchanov/XMM/sasval19/spec_merge/circinus/0111240101/works/m1_src_spec.fits rmfset=/lhome/ivaltchanov/XMM/sasval19/spec_merge/circinus/0111240101/works/m1_src_spec.rmf withenergybins=yes energymin=0.1 energymax=12.0 nenergybins=2400 returned 0


Running arfgen, can take time...


Execution of arfgen spectrumset=/lhome/ivaltchanov/XMM/sasval19/spec_merge/circinus/0111240101/works/m1_src_spec.fits withrmfset=yes rmfset=/lhome/ivaltchanov/XMM/sasval19/spec_merge/circinus/0111240101/works/m1_src_spec.rmf badpixlocation=P0111240101M1S001MIEVLI0000.FTZ arfset=/lhome/ivaltchanov/XMM/sasval19/spec_merge/circinus/0111240101/works/m1_src_spec.arf returned 0


Running rmfgen, can take time...


...

Execution of arfgen spectrumset=/lhome/ivaltchanov/XMM/sasval19/spec_merge/circinus/0111240101/works/m2_src_spec.fits withrmfset=yes rmfset=/lhome/ivaltchanov/XMM/sasval19/spec_merge/circinus/0111240101/works/m2_src_spec.rmf badpixlocation=P0111240101M2S002MIEVLI0000.FTZ arfset=/lhome/ivaltchanov/XMM/sasval19/spec_merge/circinus/0111240101/works/m2_src_spec.arf returned 0


Running rmfgen, can take time...


Execution of rmfgen spectrumset=/lhome/ivaltchanov/XMM/sasval19/spec_merge/circinus/0111240101/works/pn_src_spec.fits rmfset=/lhome/ivaltchanov/XMM/sasval19/spec_merge/circinus/0111240101/works/pn_src_spec.rmf acceptchanrange=yes withenergybins=yes energymin=0.1 energymax=12.0 nenergybins=2400 returned 0


Running arfgen, can take time...


Execution of arfgen spectrumset=/lhome/ivaltchanov/XMM/sasval19/spec_merge/circinus/0111240101/works/pn_src_spec.fits withrmfset=yes rmfset=/lhome/ivaltchanov/XMM/sasval19/spec_merge/circinus/0111240101/works/pn_src_spec.rmf badpixlocation=P0111240101PNS003PIEVLI0000.FTZ arfset=/lhome/ivaltchanov/XMM/sasval19/spec_merge/circinus/0111240101/works/pn_src_spec.arf returned 0


## Append the file names for merging

In [27]:
#
os.chdir(out_dir)
pha = ""
bkg = ""
rmf = ""
arf = ""
for inst in ['m1','m2','pn']:
    spec_src_set = f"{inst}_src_spec.fits"
    spec_bkg_set = f"{inst}_bkg_spec.fits"
    rmfset = f"{inst}_src_spec.rmf"
    arfset = f"{inst}_src_spec.arf"
    pha += (spec_src_set + " ")
    bkg += (spec_bkg_set + " ")
    rmf += (rmfset + " ")
    arf += (arfset + " ")

## Merge the spectra

In [28]:
#
# 
#
comm = f'epicspeccombine pha=\"{pha.strip()}\" bkg=\"{bkg.strip()}\" rmf=\"{rmf.strip()}\" arf=\"{arf.strip()}\"' + \
    f" filepha=merged_spec.fits filebkg=merged_bkg.fits filersp=merged_resp.fits"
print ('Running epicspeccombine...')
status,xerr = st19.run_command(comm)
if (status != 0):
    print (xerr.stdout.decode())
    raise RuntimeError


Running epicspeccombine...


Execution of epicspeccombine pha="m1_src_spec.fits m2_src_spec.fits pn_src_spec.fits" bkg="m1_bkg_spec.fits m2_bkg_spec.fits pn_bkg_spec.fits" rmf="m1_src_spec.rmf m2_src_spec.rmf pn_src_spec.rmf" arf="m1_src_spec.arf m2_src_spec.arf pn_src_spec.arf" filepha=merged_spec.fits filebkg=merged_bkg.fits filersp=merged_resp.fits returned 0


### Add some FITS keywords for background, response and ancillary to ease the reading

BACKFILE, RESPFILE, and ANCRFILE 

In [29]:
for xinst in ['m1','m2','pn']:
    sp = f"{xinst}_src_spec.fits"
    bk = f"{xinst}_bkg_spec.fits"
    ar = f"{xinst}_src_spec.arf"
    rm = f"{xinst}_src_spec.rmf"
    with fits.open(sp, mode='update') as hdu:
        hdu['SPECTRUM'].header['BACKFILE'] = bk
        hdu['SPECTRUM'].header['RESPFILE'] = rm
        hdu['SPECTRUM'].header['ANCRFILE'] = ar
#
# and the merged spectrum
# filepha=merged_spec.fits filebkg=merged_bkg.fits filersp=merged_resp.fits
with fits.open('merged_spec.fits',mode='update') as hdu:
    hdu['SPECTRUM'].header['BACKFILE'] = 'merged_bkg.fits'
    hdu['SPECTRUM'].header['RESPFILE'] = 'merged_resp.fits'
#